# 32,487 real A/B tests — what honest readouts change

Narrative companion to the repo. Everything here reads the **precomputed
exploratory results** (`make analysis`); the confirmatory + holdout numbers in
the README come from the one-shot final run and are not recomputed here.

The archive: every headline/image test Upworthy fielded, Jan 2013 – Apr 2015.
Outcomes are binomial (clicks / impressions, CTR ≈ 1.5%). Methods were
developed on the exploratory split only.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))

from analysis.paths import RESULTS_DIR, FIGURES_DIR

tests = pd.read_parquet(RESULTS_DIR / "exploratory_tests.parquet")
arms = pd.read_parquet(RESULTS_DIR / "exploratory_arms.parquet")
summary = json.loads((RESULTS_DIR / "exploratory_summary.json").read_text())
f"{len(tests):,} tests / {len(arms):,} arms loaded" 

## 1. Most experiments could not answer the question they asked

Every test gets one of five verdicts. Health checks come first (a test whose
traffic split is broken is *invalid*, not "significant"), then minimum sample,
then Holm-corrected comparisons, then a power gate against the lifts that
actually occur in this corpus.

In [ ]:
verdicts = tests["verdict"].value_counts()
display(verdicts.to_frame("tests").assign(share=(verdicts / len(tests)).round(3)))
print(f"median achieved power vs a corpus-typical lift: "
      f"{summary['power']['median_achieved_power']:.0%}")

## 2. The winner's curse, measured on real experiments

The arm you notice *because it won* is exaggerated by selection. We fit a prior
on true lifts from the corpus (Normal on log-odds lift, marginal MLE), shrink
each observed winner, and compare.

In [ ]:
from IPython.display import Image
wc = summary["winners_curse"]
print(f"corrected winners: {wc['n_corrected_winners']}\n"
      f"median naive lift: {wc['median_naive_rel_lift']:+.0%}  |  "
      f"median corrected: {wc['median_shrunk_rel_lift']:+.0%}\n"
      f"median exaggeration: x{wc['median_exaggeration_ratio']:.2f}")
Image(FIGURES_DIR / "exploratory" / "winners_curse.png")

## 3. One experiment, read out properly

The same `abkit` readout the dashboard renders. Note the order: SRM and sample
gates *before* any lift; Holm-corrected p-values; naive **and** shrunk lift;
Bayesian P(best)/expected loss labeled as decision aids, not significance.

In [ ]:
from abkit.readout import ArmCounts, analyze_experiment
from analysis.config import load_config

cfg = load_config()
example_id = tests.loc[tests["verdict"] == "ship_variant", "test_id"].iloc[0]
g = arms[arms["test_id"] == example_id].sort_values("arm_idx")
readout = analyze_experiment(
    [ArmCounts(f"arm {r.arm_idx}", int(r.impressions), int(r.clicks))
     for r in g.itertuples()],
    cfg.readout,
)
print(readout.headline, "\n")
for note in readout.notes:
    print("-", note)

## 4. Peeking: the failure mode sequential methods exist for

Left: on simulated null streams at corpus-realistic sizes, stop-at-first-p<.05
inflates false positives far above 5%; mSPRT monitored at *every* look stays
below it. On conditional-permutation replays of the real tests, naive peeking
would have declared a winner mid-test in ~37% of the experiments whose full
data show no corrected winner.

In [ ]:
pk = summary["peeking"]
print("naive false-positive rate by number of looks:", pk["sim_naive_fp_by_looks"])
print(f"mSPRT under continuous monitoring: {pk['sim_msprt_fp_continuous']:.1%}")
print(f"replayed phantom-win rate  naive: {pk['replay_phantom_win_rate_naive']:.0%}"
      f"  |  mSPRT: {pk['replay_phantom_win_rate_msprt']:.0%}")
Image(FIGURES_DIR / "exploratory" / "peeking_damage.png")

## 5. Honest accounting, in one chart each

- **FDR**: how many "wins" survive within-test Holm and then corpus-level BH.
- **SRM**: the share of tests whose traffic split is inconsistent with uniform
  randomization (excluded from every win count above).
- **Upworthy's own calls**: only ~1 in 5 of their declared winners is confirmed
  by the corrected analysis — and half were declared from underpowered tests.

In [ ]:
ua = summary["upworthy_audit"]
print(f"Upworthy declared {ua['n_declared']:,} winners "
      f"({ua['frac_of_tests']:.0%} of tests); "
      f"{ua['frac_confirmed_by_corrected_analysis']:.0%} confirmed, "
      f"{ua['frac_underpowered_verdict']:.0%} from underpowered tests, "
      f"{ua['frac_srm_failed']:.0%} from SRM-failing tests.")
display(Image(FIGURES_DIR / "exploratory" / "fdr_accounting.png"))
display(Image(FIGURES_DIR / "exploratory" / "srm_audit.png"))

## Where to go next

- `src/abkit/` — the library; every claim above is backed by a simulation test
  in `tests/`.
- `make dashboard` — the interactive readout tool.
- `README.md` — headline (confirmatory + holdout) numbers, methods appendix,
  limitations, and what this would need inside a real experimentation platform.